# Density matrices
**Quantum Computing · Week 6 (Noise and real hardware) · Day 2**

Lesson page: https://sara-aminpour.com/tutorials/qc/w6d2.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

A **density matrix** $\rho$ describes everything we can know about a quantum state, pure or mixed.
For a pure state $\rho = |\psi\rangle\langle\psi|$; for a mixture, $\rho = \sum_k p_k |\psi_k\rangle\langle\psi_k|$.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, DensityMatrix, partial_trace, Pauli

theta, phi = 1.2, 0.8
psi = np.array([np.cos(theta / 2), np.exp(1j * phi) * np.sin(theta / 2)])
rho = np.outer(psi, psi.conj())                       # |psi><psi|: a column times a row
print("rho =\n", np.round(rho, 3))
print("same as Qiskit:", np.allclose(rho, DensityMatrix(Statevector(psi)).data))
print("trace:", np.round(np.trace(rho).real, 3), "  Hermitian:", np.allclose(rho, rho.conj().T),
      "  eigenvalues:", np.round(np.linalg.eigvalsh(rho), 3) + 0.0, "  purity Tr(rho^2):", np.round(np.trace(rho @ rho).real, 3))

The diagonal holds the probabilities of 0 and 1; the off-diagonal entries (the **coherences**) hold the
relative phase. A pure state has eigenvalues 1 and 0, and purity 1.

## Two recipes, one matrix
Mix $|0\rangle$ and $|1\rangle$ half and half, or $|+\rangle$ and $|-\rangle$ half and half.

In [ ]:
def dm(label):
    return DensityMatrix.from_label(label).data

recipe_1 = 0.5 * dm("0") + 0.5 * dm("1")
recipe_2 = 0.5 * dm("+") + 0.5 * dm("-")
print("0/1 recipe:\n", np.round(recipe_1.real, 3))
print("+/- recipe:\n", np.round(recipe_2.real, 3) + 0.0)
print("identical:", np.allclose(recipe_1, recipe_2), "  purity:", np.trace(recipe_1 @ recipe_1).real)

Two different ways of preparing the qubit, one and the same $\rho = I/2$. Since every measurement
statistic is computed from $\rho$, **no experiment can tell the recipes apart**.

## Everything comes from rho
Probabilities are diagonal entries, expectation values are $\langle O\rangle = \mathrm{Tr}(O\rho)$,
and a gate $U$ acts as $\rho \to U \rho U^\dagger$.

In [ ]:
mixed = DensityMatrix(0.7 * dm("0") + 0.3 * dm("+"))
print("probabilities of 0 and 1:", mixed.probabilities().round(3))
bloch = [np.trace(Pauli(p).to_matrix() @ mixed.data).real for p in "XYZ"]
print("Bloch vector (Tr(X rho), Tr(Y rho), Tr(Z rho)):", np.round(bloch, 3) + 0.0, "  length:", round(float(np.linalg.norm(bloch)), 3))
h_gate = QuantumCircuit(1)
h_gate.h(0)
print("after an H gate:", np.round([np.trace(Pauli(p).to_matrix() @ mixed.evolve(h_gate).data).real for p in "XYZ"], 3) + 0.0)

## Pieces of an entangled pair
The **partial trace** keeps one qubit and averages over the other. For a Bell pair the whole is pure,
yet each half alone is the completely mixed state $I/2$: entanglement makes the parts look like noise.
(That's what noise often is: entanglement with an environment we can't see.)

In [ ]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
whole = DensityMatrix(bell)
half = partial_trace(whole, [1])                      # keep qubit 0, trace out qubit 1
print("purity of the pair:", round(float(whole.purity().real), 3))
print("qubit 0 alone:\n", np.round(half.data.real, 3) + 0.0, "  purity:", round(float(half.purity().real), 3))

## Your turn: an entanglement meter
In week 4, $R_Y(\theta)$ on qubit 0 followed by a CNOT dialed up the entanglement. Compute the purity of
qubit 0 alone for $\theta = 0°, 30°, 60°, 90°$. What purity means "not entangled", and what means "maximally"?

In [ ]:
for degrees in [0, 30, 60, 90]:
    qc = QuantumCircuit(2)
    qc.ry(np.radians(degrees), 0)
    qc.cx(0, 1)
    purity = 1.0     # TODO: partial_trace(DensityMatrix(qc), [1]).purity().real
    print(degrees, purity)

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
for degrees in [0, 30, 60, 90]:
    qc = QuantumCircuit(2)
    qc.ry(np.radians(degrees), 0)
    qc.cx(0, 1)
    purity = partial_trace(DensityMatrix(qc), [1]).purity().real
    print(f"theta = {degrees:2d} degrees: purity of qubit 0 alone = {purity:.3f}")
print("Purity 1: a product state, no entanglement. Purity 1/2: maximally entangled, each half a fair coin.")